# Week 13｜API key 設定與檢查（停課週作業）
**API key setup check — 12/6（日）23:59 前 push**

第 14 週要做期末系統的最後一塊：**把第 12 週找到的條文交給語言模型，請它回答**。呼叫語言模型需要每個人自己的 **API key**。這份 notebook 幫你確認：

1. 套件已安裝
2. `.env` 不會被上傳到 GitHub（**最重要**）
3. API key 可以用
4. 直接問語言模型校規問題，會發生什麼事？（第 14 週為什麼要先「檢索」）

**要繳交：** 這份 notebook（保留輸出）、`week13_api_check.json`、`AI_USAGE.md` 的 Week 13 段落、更新後的 `.gitignore`、`requirements.txt`。

---

## 步驟 1｜建立 Groq 帳號與 API key（在瀏覽器）

1. 到 **https://console.groq.com** 註冊並登入（可以用 Google 或 GitHub 帳號；免費，不需要信用卡）
2. 點畫面**上方選單**的 **API Keys**（在 Playground 和 Dashboard 中間）
3. 點右邊的 **＋ Create API Key** → 名稱填 `nlp-course`
   - 如果畫面要你選**到期日 (Expiration)**：請選 2027 年 1 月以後，或「不會到期」，避免期末前失效
4. 按下建立後，**馬上複製** key（以 `gsk_` 開頭）。**只會顯示這一次**，先貼在記事本，等一下要用
   - 忘了複製也沒關係：刪掉那把 key，再建立一把新的

## 步驟 2｜安裝套件（VS Code 終端機，確認有 `(.venv)`）

```
pip install openai python-dotenv
```

## 步驟 3｜執行下一格：自動把 `.env` 加進 `.gitignore`

**一定要先做這一步，才建立 `.env`。**

In [ ]:
import os, re, sys, json, subprocess, importlib.util
from pathlib import Path
from datetime import datetime

notebook_dir = os.path.abspath('')
# 往上推兩層取得根目錄（notebook 放在 notebooks/week13/）
#project_root = os.path.dirname(os.path.dirname(notebook_dir))
# 往上推一層（notebook 放在 notebooks/）→ 把上一行加上 #，這一行的 # 拿掉
project_root = os.path.dirname(notebook_dir)
ROOT = Path(project_root)
assert (ROOT / "data").exists(), f"{ROOT} 不像專案根目錄：請檢查上面是往上推兩層還是一層"

missing = [p for p, m in [("openai", "openai"), ("python-dotenv", "dotenv")] if importlib.util.find_spec(m) is None]
assert not missing, f"找不到套件 {missing}：請在終端機（確認有 (.venv)）執行 pip install openai python-dotenv，並確認右上角 kernel 是 .venv"

gi = ROOT / ".gitignore"
lines = gi.read_text(encoding="utf-8").splitlines() if gi.exists() else []
if ".env" not in [l.strip() for l in lines]:
    gi.write_text("\n".join(lines).rstrip("\n") + "\n\n# Week 13：API key 不可上傳\n.env\n", encoding="utf-8")
    print("已在 .gitignore 加入 .env")

req = ROOT / "requirements.txt"
req_lines = req.read_text(encoding="utf-8").splitlines() if req.exists() else []
new_req = [p for p in ["openai", "python-dotenv"] if p not in [l.strip() for l in req_lines]]
if new_req:
    req.write_text("\n".join(req_lines + new_req).strip("\n") + "\n", encoding="utf-8")
print("✅ 套件已安裝；.gitignore 有 .env；requirements.txt 有 openai、python-dotenv")

## 步驟 4｜建立 `.env` 檔

1. VS Code 左側檔案總管，在**專案根目錄**（和 `README.md` 同一層）按右鍵 → **New File** → 檔名輸入 `.env`（前面有一個點，沒有其他字）
2. 內容只有一行（**不要加引號、不要有空白**）：

```
GROQ_API_KEY=gsk_你複製的key
```

3. 存檔（`Ctrl + S`）。存好之後，`.env` 在檔案總管中應該是**灰色**的：表示 Git 會忽略它。

然後執行下一格。

In [ ]:
ENV = ROOT / ".env"
assert ENV.exists(), f"找不到 {ENV}：請在專案根目錄建立 .env（檔名前面有一個點）。Windows 注意：檔名不可以是 .env.txt"

from dotenv import load_dotenv
load_dotenv(ENV, override=True)
API_KEY = os.getenv("GROQ_API_KEY", "")
assert API_KEY, ".env 裡面找不到 GROQ_API_KEY：內容應該是一行 GROQ_API_KEY=gsk_..."
assert API_KEY == API_KEY.strip().strip('"').strip("'"), ".env 中的 key 前後不可以有引號或空白"
assert API_KEY.startswith("gsk_"), "Groq 的 key 應該以 gsk_ 開頭：請重新複製"

# Git 安全檢查：.env 必須被忽略、而且從來沒有被加入過
def git(*args):
    return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True)
try:
    ignored = git("check-ignore", "-q", ".env").returncode == 0
    tracked = bool(git("ls-files", ".env").stdout.strip())
except FileNotFoundError:
    ignored, tracked = None, None
    print("（這台電腦的終端機找不到 git 指令，略過 Git 檢查；請用 VS Code 的原始檔控制確認 .env 沒有出現在變更清單）")
assert tracked is not True, (".env 已經被 Git 追蹤！請在終端機執行：git rm --cached .env\n"
                             "如果已經 push 過：到 console.groq.com → API Keys 刪除這把 key，再建立新的。")
assert ignored is not False, ".env 沒有被 Git 忽略：請確認 .gitignore 有一行 .env，然後重新執行"
print("✅ 讀到 API key（只顯示前 8 碼）：", API_KEY[:8] + "…")
print("✅ .env 不會被上傳到 GitHub")

## 步驟 5｜呼叫語言模型

`MODEL` 使用老師指定的模型（老師公告如有更換，只要改這一行）。

In [ ]:
from openai import OpenAI, AuthenticationError, RateLimitError, APIConnectionError, NotFoundError, BadRequestError

BASE_URL = "https://api.groq.com/openai/v1"
MODEL = "openai/gpt-oss-120b"          # 老師指定的模型

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

def ask(question, max_tokens=500):
    extra = {"reasoning_effort": "low"} if "gpt-oss" in MODEL else None
    messages = [{"role": "system", "content": "請用繁體中文（台灣用語）簡短回答。"},
                {"role": "user", "content": question}]
    for _ in range(2):
        try:
            r = client.chat.completions.create(model=MODEL, max_tokens=max_tokens, temperature=0.2,
                                               extra_body=extra, messages=messages)
            break
        except AuthenticationError:
            raise SystemExit("❌ API key 無效：請到 console.groq.com → API Keys 重新建立一把，貼到 .env，再從步驟 4 重新執行")
        except RateLimitError:
            raise SystemExit("❌ 超過額度（免費帳號每分鐘、每天都有上限）：等一分鐘再執行")
        except APIConnectionError:
            raise SystemExit("❌ 連不上 Groq：請檢查網路，或換一個網路再試")
        except (NotFoundError, BadRequestError) as e:
            if extra:              # 這個模型不支援 reasoning_effort → 拿掉再試一次
                extra = None
                continue
            raise SystemExit(f"❌ 模型名稱 {MODEL} 可能已經停用：請看老師最新公告，修改 MODEL。（{e}）")
    answer = re.sub(r"<think>.*?</think>", "", r.choices[0].message.content or "", flags=re.S).strip()
    return answer, r.usage

answer, usage = ask("用一句話說明什麼是 API key。")
print(answer)
print(f"\n（用了 {usage.prompt_tokens} + {usage.completion_tokens} 個 token）")
assert answer, "沒有收到回答，請再執行一次"
print("\n✅ 語言模型可以用")

## 步驟 6｜直接問語言模型校規問題，會怎樣？

下一格**不給任何條文**，直接問模型我們學校的規定。正確答案來自第 12 週的問答集：

| 問題 | 校規的答案 |
|---|---|
| 每學期申請停修科目以幾科為限？ | 2 科（學生申請期中停修課程實施要點 第二條） |
| 心理健康假一學期最多可以請幾天？ | 7 天（學生請假作業辦法 第四條） |

In [ ]:
CHECKS = [("每學期申請停修科目以幾科為限？", "2 科"),
          ("心理健康假一學期最多可以請幾天？", "7 天")]
no_context = []
for q, truth in CHECKS:
    a, _ = ask("我是明志科技大學的學生。" + q)
    no_context.append({"question": q, "truth": truth, "model_answer": a})
    print(f"問：{q}\n模型：{a}\n校規：{truth}\n")

### 反思（2–3 句，直接在這一格作答）

模型的回答和校規一樣嗎？它有沒有說「不知道」，還是很有自信地給了一個答案？這說明了第 14 週為什麼要**先檢索條文、再請模型根據條文回答**？

> （在這裡作答）

In [ ]:
my_reflection = ""   # TODO：把上面的反思也寫在這裡（一兩句）

assert my_reflection.strip(), "請在 my_reflection 寫下你的反思"
out = {
    "week": 13,
    "model": MODEL,
    "checked_at": datetime.now().isoformat(timespec="seconds"),
    "api_ok": True,
    "env_ignored_by_git": ignored,
    "no_context_answers": no_context,
    "reflection": my_reflection,
}
text = json.dumps(out, ensure_ascii=False, indent=2)
assert API_KEY not in text, "JSON 裡面不可以有 API key"
OUT = ROOT / "week13_api_check.json"
OUT.write_text(text, encoding="utf-8")
print("✅ 已儲存", OUT)

## 繳交（12/6 23:59 前）

- [ ] 所有 ✅ 都出現，notebook **保留輸出**並存檔
- [ ] 反思已作答
- [ ] `AI_USAGE.md` 新增 **Week 13** 段落
- [ ] `git status`：**不應該出現 `.env`**
- [ ] commit & push，並打開 GitHub 網頁確認

```bash
git status
git add notebooks/week13/13_api_check.ipynb week13_api_check.json AI_USAGE.md .gitignore requirements.txt
git commit -m "Week 13: API key setup check"
git push
```

或用 VS Code **原始檔控制（樹枝狀圖示）**：確認清單中**沒有 `.env`** → [ + ] 暫存 → 輸入訊息 → Commit → Sync Changes

### API key 安全守則

- **不要**把 key 寫進 notebook、`AI_USAGE.md`，或貼到任何 AI 對話中
- **不要**和同學共用 key：額度是每個帳號分開計算的
- 不小心 push 了 `.env`：到 console.groq.com → API Keys **刪除那把 key**，再建立新的，並執行 `git rm --cached .env`